In [ ]:
import shutil
from langchain_core.documents import Document

def generate_question(state: InterviewState) -> InterviewState:

    """
      질문 생성 (고도화 버전)
        - next_step이 new_question이면 새로운 주제의 질문 생성
        - next_step이 additional_question이면 답변 기반 심화 질문 생성
        - Chroma Vector DB에서 유사 질문 3개를 검색하여 LLM 프롬프트에 참고로 포함

    """

    # LLM 및 임베딩 모델 정의
    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.6)
    embedding = OpenAIEmbeddings(model="text-embedding-3-small")

    # 유사 질문 생성 프롬프트
    sub_prompt = ChatPromptTemplate.from_template("""
      당신은 면접관입니다. 아래 대화 맥락을 기반으로 유사한 면접 질문을 3개 생성하세요.
      각 질문은 한 문장으로, 자연스러운 한국어로 작성합니다.

      --- 이력서 요약 ---
      {resume_summary}

      --- 주요 키워드 ---
      {resume_keywords}

      --- 최근 질문 ---
      {current_question}

      --- 현재 질문 전략 ---
      {current_strategy}

      --- 지원자 답변 ---
      {current_answer}

      출력 형식:
      1) ...
      2) ...
      3) ...
    """)

    response = (sub_prompt | llm).invoke({
        "resume_summary": state.get("resume_summary", ""),
        "resume_keywords": state.get("resume_keywords", []),
        "current_question": state.get("current_question", ""),
        "current_answer": state.get("current_answer", ""),
        "current_strategy": state.get("current_strategy","")

    })

    # 결과 파싱
    lines = [line.strip("123). ").strip() for line in response.content.split("\n") if line.strip()]
    similar_questions = [q for q in lines if q and len(q) > 3][:3]


    # 벡터 DB 생성
    docs = [Document(page_content=q) for q in similar_questions]
    vectorstore = Chroma.from_documents(docs, embedding, persist_directory=None)
    retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

    # 최종 질문 생성

    similar_docs = retriever.invoke(state.get("current_answer", ""))
    similar_texts = "\n".join([f"- {doc.page_content}" for doc in similar_docs])

    main_prompt = ChatPromptTemplate.from_template("""
    당신은 면접관입니다. 아래 정보를 기반으로 지원자에게 추가로 물어볼 질문을 생성하세요.

    --- 이력서 요약 ---
    {resume_summary}

    --- 주요 키워드 ---
    {resume_keywords}

    --- 질문 전략 ---
    {question_strategy}

    --- 현재 질문 전략 ---
    {current_strategy}

    --- 이전 질문 ---
    {current_question}

    --- 지원자 답변 ---
    {current_answer}

    --- 답변 평가 ---
    {evaluation}

    --- 심화 질문 여부 ---
    {next_step}

    --- 참고용 유사 질문 ---
    {similar_examples}

    생성 규칙:
    1. 지원자의 사고력·문제 해결 방식을 파악할 수 있는 질문을 작성합니다.
    2. 한 문장, 자연스러운 한국어 질문 형태로 작성합니다.
    3. 어조는 전문적이며 격식체를 유지합니다.
    4. 'new_question'이면 현재 질문 전략(current_strategy)을 활용해 새로운 분야의 질문을 생성합니다.
    5. 'additional_question'이면 답변을 기반으로 심화 질문을 생성합니다.
    6. 모든 질문은 30자 이내로 간결하게 작성하세요.
    7. 강조기호(** 등)나 이모티콘은 사용하지 마세요.

    """)


    chain = main_prompt | llm
    response = chain.invoke({
        "resume_summary": state.get("resume_summary", ""),
        "resume_keywords": state.get("resume_keywords", []),
        "question_strategy": state.get("question_strategy", {}),
        "current_strategy" : state.get("current_strategy",""),
        "current_question": state.get("current_question", ""),
        "current_answer": state.get("current_answer", ""),
        "evaluation": state.get("evaluation", []),
        "next_step": state.get("next_step",""),
        "similar_examples": similar_texts
    })

    next_question = response.content.strip()

    return {
        **state,
        "current_question": next_question,
        "current_answer": "",
    }
